In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# TVP-VAR -> TVP-VMA -> GFEVD
# ------------------------------------------------------------
# Input
#   - tvpvar_beta.npy
#   - tvpvar_cov.npy
#   - tvpvar_selected_lag.txt
#
# Output
#   - gfevd_last.csv
#   - gfevd_mean.csv
#   - gfevd_all.npy
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")

BETA_FILE = BASE_DIR / "tvpvar_beta.npy"
COV_FILE = BASE_DIR / "tvpvar_cov.npy"
LAG_FILE = BASE_DIR / "tvpvar_selected_lag.txt"

OUT_LAST = BASE_DIR / "gfevd_last.csv"
OUT_MEAN = BASE_DIR / "gfevd_mean.csv"
OUT_ALL = BASE_DIR / "gfevd_all.npy"

H = 10  # forecast horizon

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

EPS = 1e-12


# -----------------------------
# Helpers
# -----------------------------
def make_psd(mat: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """
    공분산행렬을 대칭 + PSD 형태로 보정
    """
    mat = 0.5 * (mat + mat.T)

    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, eps, None)
    mat_psd = vecs @ np.diag(vals) @ vecs.T
    mat_psd = 0.5 * (mat_psd + mat_psd.T)
    return mat_psd


def load_inputs():
    beta_t = np.load(BETA_FILE)   # shape: (T_eff, k, k*p)
    cov_t = np.load(COV_FILE)     # shape: (T_eff, k, k)

    with open(LAG_FILE, "r", encoding="utf-8") as f:
        p = int(f.read().strip())

    if beta_t.ndim != 3:
        raise ValueError(f"beta_t shape 이상: {beta_t.shape}")
    if cov_t.ndim != 3:
        raise ValueError(f"cov_t shape 이상: {cov_t.shape}")

    T_eff, k, kp = beta_t.shape

    if p < 1:
        raise ValueError(
            "선택된 lag가 0입니다. GFEVD는 TVP-VAR(p>=1)에서만 계산 가능합니다. "
            "TVP-VAR을 다시 추정할 때 최소 lag가 1 이상이 되도록 설정하세요."
        )

    if kp != k * p:
        raise ValueError(
            f"beta_t의 마지막 차원({kp})과 k*p({k*p})가 일치하지 않습니다."
        )

    return beta_t, cov_t, p, T_eff, k


def unpack_A_mats(beta_one_t: np.ndarray, p: int) -> list:
    """
    beta_one_t: shape (k, k*p)
    return: [A1, A2, ..., Ap], each shape (k, k)
    """
    k, kp = beta_one_t.shape
    A_list = []
    for lag in range(p):
        start = lag * k
        end = (lag + 1) * k
        A_list.append(beta_one_t[:, start:end])
    return A_list


def compute_phi_list(A_list: list, H: int) -> list:
    """
    TVP-VAR 계수 A1..Ap로부터
    Phi_0 ... Phi_{H-1} 계산

    VAR:
        y_t = A1 y_{t-1} + ... + Ap y_{t-p} + e_t

    VMA:
        y_t = sum_{h=0}^\infty Phi_h e_{t-h}
    """
    p = len(A_list)
    k = A_list[0].shape[0]

    phi = [np.eye(k)]
    for h in range(1, H):
        phi_h = np.zeros((k, k))
        max_lag = min(h, p)
        for j in range(1, max_lag + 1):
            phi_h += A_list[j - 1] @ phi[h - j]
        phi.append(phi_h)

    return phi


def compute_gfevd_one_t(A_list: list, Sigma_t: np.ndarray, H: int) -> np.ndarray:
    """
    Generalized FEVD
    반환 shape: (k, k)
    행 = response i
    열 = shock source j
    """
    Sigma_t = make_psd(Sigma_t)

    k = Sigma_t.shape[0]
    phi_list = compute_phi_list(A_list, H)

    gfevd = np.zeros((k, k))

    for i in range(k):
        e_i = np.zeros((k, 1))
        e_i[i, 0] = 1.0

        denom = 0.0
        for h in range(H):
            phi_h = phi_list[h]
            term = e_i.T @ phi_h @ Sigma_t @ phi_h.T @ e_i
            denom += float(term)

        denom = max(denom, EPS)

        for j in range(k):
            e_j = np.zeros((k, 1))
            e_j[j, 0] = 1.0

            sigma_jj = float(Sigma_t[j, j])
            sigma_jj = max(sigma_jj, EPS)

            numer = 0.0
            for h in range(H):
                phi_h = phi_list[h]
                term = e_i.T @ phi_h @ Sigma_t @ e_j
                numer += float(term) ** 2

            gfevd[i, j] = (1.0 / sigma_jj) * (numer / denom)

    # row normalization
    row_sums = gfevd.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums < EPS, 1.0, row_sums)
    gfevd = gfevd / row_sums

    return gfevd


# -----------------------------
# Main
# -----------------------------
def main():
    beta_t, cov_t, p, T_eff, k = load_inputs()

    if len(VAR_NAMES) != k:
        raise ValueError(
            f"VAR_NAMES 길이({len(VAR_NAMES)})와 k({k})가 다릅니다."
        )

    print("==========================================")
    print("TVP-VAR -> GFEVD")
    print("==========================================")
    print("T_eff :", T_eff)
    print("k     :", k)
    print("lag p :", p)
    print("H     :", H)
    print()

    gfevd_all = np.zeros((T_eff, k, k))

    for t in range(T_eff):
        A_list = unpack_A_mats(beta_t[t], p)
        Sigma_t = cov_t[t]

        gfevd_t = compute_gfevd_one_t(A_list, Sigma_t, H)
        gfevd_all[t] = gfevd_t

        if (t + 1) % 200 == 0 or (t + 1) == T_eff:
            print(f"processed: {t + 1}/{T_eff}")

    # 마지막 시점
    gfevd_last = gfevd_all[-1]
    df_last = pd.DataFrame(gfevd_last, index=VAR_NAMES, columns=VAR_NAMES)

    # 전체 평균
    gfevd_mean = gfevd_all.mean(axis=0)
    df_mean = pd.DataFrame(gfevd_mean, index=VAR_NAMES, columns=VAR_NAMES)

    # 저장
    df_last.to_csv(OUT_LAST, encoding="utf-8-sig")
    df_mean.to_csv(OUT_MEAN, encoding="utf-8-sig")
    np.save(OUT_ALL, gfevd_all)

    print()
    print("Saved files:")
    print(f" - {OUT_LAST}")
    print(f" - {OUT_MEAN}")
    print(f" - {OUT_ALL}")
    print()

    print("Last-period GFEVD:")
    print(df_last.round(4).to_string())
    print()
    print("Mean GFEVD:")
    print(df_mean.round(4).to_string())


if __name__ == "__main__":
    main()

ValueError: 선택된 lag가 0입니다. GFEVD는 TVP-VAR(p>=1)에서만 계산 가능합니다. TVP-VAR을 다시 추정할 때 최소 lag가 1 이상이 되도록 설정하세요.